In [ ]:
import os
import gc
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
import json
import pickle
from datetime import datetime
from IPython.display import display
from collections import Counter
import html
import re
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    mean_absolute_error,
    confusion_matrix,
    classification_report
)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.data import Dataset as TorchDataset
from torch.nn.utils.rnn import pad_sequence

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)


# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## Exploratory Data Analysis (EDA)

Initial EDA using the training set only.  
The purpose of this stage is to understand the data distribution, missing values, label imbalance, review length, useful reviews, and basic textual patterns before any modeling or preprocessing decisions.


In [ ]:
# Load the training dataset only
TRAIN_PATH = 'drugsComTrain_raw.csv'

train = pd.read_csv(TRAIN_PATH)

print(f'Train shape: {train.shape}')
display(train.head(10))

In [ ]:
# Basic dataset overview
display(train.info())
display(train.describe(include='all'))

In [ ]:
# Missing values analysis
missing = pd.DataFrame({
    'missing_count': train.isnull().sum(),
    'missing_percent': train.isnull().mean() * 100
}).sort_values('missing_percent', ascending=False)

display(missing)

In [ ]:
# Inspect rows with missing condition values to understand whether they still contain useful information

missing_condition_rows = train[train['condition'].isnull()]

print(f'Rows with missing condition: {len(missing_condition_rows)}')
print(f'Percentage of train data: {len(missing_condition_rows) / len(train) * 100:.4f}%')

display(missing_condition_rows.head(10))

In [ ]:
# Duplicate rows check
duplicates = train.duplicated().sum()
print(f'Duplicate rows: {duplicates}')

In [ ]:
# Rating distribution table
rating_distribution = (
    train['rating']
    .value_counts()
    .sort_index()
    .rename_axis('rating')
    .reset_index(name='count')
)

rating_distribution['percent'] = rating_distribution['count'] / len(train) * 100

display(rating_distribution)

In [ ]:
# Rating distribution count plot
# The skewed distribution means accuracy alone will not be enough and other metrics should be measured

plt.figure(figsize=(10, 5))
sns.countplot(data=train, x='rating', order=sorted(train['rating'].unique()))
plt.title('Number of Reviews per Rating - Train Data Only')
plt.xlabel('Rating')
plt.ylabel('Number of Reviews')
plt.show()

In [ ]:
# Create 3-class sentiment labels for comparison with the attached paper
def rating_to_sentiment(r):
    if r <= 4:
        return 'negative'
    elif r < 7:
        return 'neutral'
    return 'positive'

train['sentiment_3class'] = train['rating'].apply(rating_to_sentiment)

sentiment_distribution = (
    train['sentiment_3class']
    .value_counts()
    .rename_axis('sentiment')
    .reset_index(name='count')
)

sentiment_distribution['percent'] = sentiment_distribution['count'] / len(train) * 100

# Sentiment distribution plot
plt.figure(figsize=(8, 5))
sns.countplot(data=train, x='sentiment_3class', order=['negative', 'neutral', 'positive'])
plt.title('3-Class Sentiment Distribution - Paper-Based Mapping')
plt.xlabel('Sentiment Class')
plt.ylabel('Number of Reviews')
plt.show()

In [ ]:
# Review length analysis
# It may help later to choose max sequence length

train['review_length_chars'] = train['review'].astype(str).apply(len)
train['review_length_words'] = train['review'].astype(str).apply(lambda x: len(x.split()))

length_summary = train[['review_length_chars', 'review_length_words']].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
)

# Review length distribution
plt.figure(figsize=(10, 5))
sns.histplot(train['review_length_words'], bins=60)
plt.title('Review Length Distribution')
plt.xlabel('Number of Words')
plt.ylabel('Number of Reviews')
plt.xlim(0, 250)
plt.show()

In [ ]:
# Approximate truncation analysis using word counts
# It gives an early estimate for choosing max_length tokens

for limit in [64, 128, 256, 512]:
    percent_above = (train['review_length_words'] > limit).mean() * 100
    print(f'Reviews longer than {limit} words: {percent_above:.2f}%')

In [ ]:
# Review length by rating
# This checks whether very positive or very negative reviews tend to be longer.
plt.figure(figsize=(12, 5))
sns.boxplot(data=train, x='rating', y='review_length_words')
plt.title('Review Length by Rating')
plt.xlabel('Rating')
plt.ylabel('Review Length in Words')
plt.ylim(0, train['review_length_words'].quantile(0.99))
plt.show()

In [ ]:
# Review length by 3-class sentiment
plt.figure(figsize=(8, 5))
sns.boxplot(data=train, x='sentiment_3class', y='review_length_words', order=['negative', 'neutral', 'positive'])
plt.title('Review Length by Sentiment Class')
plt.xlabel('Sentiment Class')
plt.ylabel('Review Length in Words')
plt.ylim(0, train['review_length_words'].quantile(0.99))
plt.show()

In [ ]:
# Most frequent drugs
most_freq_drugs = train['drugName'].value_counts().head(20)
plt.figure(figsize=(10, 8))
most_freq_drugs.sort_values().plot(kind='barh')
plt.title('Top 20 Most Frequent Drugs')
plt.xlabel('Number of Reviews')
plt.ylabel('Drug Name')
plt.show()

In [ ]:
# Most frequent conditions
top_conditions = train['condition'].value_counts().head(20)
plt.figure(figsize=(10, 8))
top_conditions.sort_values().plot(kind='barh')
plt.title('Top 20 Most Frequent Conditions')
plt.xlabel('Number of Reviews')
plt.ylabel('Condition')
plt.show()

In [ ]:
# Average rating per frequent drug
drug_stats = (
    train
    .groupby('drugName')
    .agg(
        count=('rating', 'count'),
        mean_rating=('rating', 'mean'),
        median_rating=('rating', 'median')
    )
    .sort_values('count', ascending=False)
)

frequent_drug_stats = drug_stats[drug_stats['count'] >= 100].sort_values('mean_rating', ascending=False)
display(frequent_drug_stats.head(10))
display(frequent_drug_stats.tail(10))

In [ ]:
# Average rating per frequent condition
condition_stats = (
    train
    .dropna(subset=['condition'])
    .groupby('condition')
    .agg(
        count=('rating', 'count'),
        mean_rating=('rating', 'mean'),
        median_rating=('rating', 'median')
    )
    .sort_values('count', ascending=False)
)

frequent_condition_stats = condition_stats[condition_stats['count'] >= 100].sort_values('mean_rating', ascending=False)
display(frequent_condition_stats.head(10))
display(frequent_condition_stats.tail(10))

In [ ]:
# usefulCount summary
# Its distribution is highly skewed, so inspect both raw values and a log-transformed version

display(train['usefulCount'].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))
train['log_usefulCount'] = np.log1p(train['usefulCount'])
display(train['log_usefulCount'].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

In [ ]:
# usefulCount distribution
plt.figure(figsize=(10, 5))
sns.histplot(train['usefulCount'], bins=50)
plt.title('usefulCount Distribution')
plt.xlabel('usefulCount')
plt.ylabel('Number of Reviews')
plt.show()

# log_usefulCount distribution
plt.figure(figsize=(10, 5))
sns.histplot(train['log_usefulCount'], bins=50)
plt.title('Log-Transformed usefulCount Distribution')
plt.xlabel('log(1 + usefulCount)')
plt.ylabel('Number of Reviews')
plt.show()

In [ ]:
# Correlation matrix for numeric EDA features
numeric_cols = ['rating', 'usefulCount', 'log_usefulCount', 'review_length_chars', 'review_length_words']
corr_matrix = train[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='Blues')
plt.title('Correlation Matrix of Numeric Features')
plt.show()

In [ ]:
# Check repeated review texts and inconsistent labels
# If the same exact review text appears with multiple ratings, this may indicate label noise.
review_label_noise = (
    train
    .groupby('review')
    .agg(
        count=('rating', 'count'),
        unique_ratings=('rating', 'nunique'),
        ratings_list=('rating', lambda x: sorted(x.unique()))
    )
    .sort_values(['unique_ratings', 'count'], ascending=False)
)

repeated_reviews = review_label_noise[review_label_noise['count'] > 1]
inconsistent_repeated_reviews = repeated_reviews[repeated_reviews['unique_ratings'] > 1]

print(f'Number of exact repeated review texts: {len(repeated_reviews)}')
print(f'Number of repeated review texts with inconsistent ratings: {len(inconsistent_repeated_reviews)}')

display(inconsistent_repeated_reviews.head(20))

In [ ]:
# Top unigrams by sentiment class
def get_top_ngrams(texts, ngram_range=(1, 1), top_n=20, min_df=5):
    vectorizer = CountVectorizer(
        stop_words='english',
        ngram_range=ngram_range,
        min_df=min_df,
        max_features=5000
    )
    X = vectorizer.fit_transform(texts.astype(str))
    counts = np.asarray(X.sum(axis=0)).ravel()
    features = vectorizer.get_feature_names_out()

    return (
        pd.DataFrame({'term': features, 'count': counts})
        .sort_values('count', ascending=False)
        .head(top_n)
    )

negative_unigrams = get_top_ngrams(train[train['sentiment_3class'] == 'negative']['review'], ngram_range=(1, 1), top_n=20)
positive_unigrams = get_top_ngrams(train[train['sentiment_3class'] == 'positive']['review'], ngram_range=(1, 1), top_n=20)

print('Top negative unigrams:')
display(negative_unigrams)

print('\nTop positive unigrams:')
display(positive_unigrams)

In [ ]:
# Top bigrams by sentiment class
negative_bigrams = get_top_ngrams(train[train['sentiment_3class'] == 'negative']['review'], ngram_range=(2, 2), top_n=20)
positive_bigrams = get_top_ngrams(train[train['sentiment_3class'] == 'positive']['review'], ngram_range=(2, 2), top_n=20)

print('Top negative bigrams:')
display(negative_bigrams)

print('\nTop positive bigrams:')
display(positive_bigrams)

## Preprocessing

In [ ]:
# Randomly inspect 10 reviews from the training set to help manually identify text artifacts
# such as HTML entities, strange characters, encoding issues, repeated patterns, or noisy reviews
sample_reviews = train.sample(10, random_state=42)

for idx, row in sample_reviews.iterrows():
    print(f"Index: {idx}")
    print(f"Drug: {row['drugName']}")
    print(f"Condition: {row['condition']}")
    print(f"Rating: {row['rating']}")
    print(f"Useful Count: {row['usefulCount']}")
    print(row["review"])
    print()

In [ ]:
# Work on a copy of the original training data
df = train.copy()

# Sanity check: make sure the copy has the same shape as the original training set
print(f"Original train shape: {train.shape}")
print(f"Working copy shape: {df.shape}")

In [ ]:
# Handle missing condition values
# Since only 0.56% of the training data has missing condition values, I decided to fill them with
# 'Unknown', to preserve all samples and explicitly represent missing condition information
missing_before = df["condition"].isnull().sum()
df["condition"] = df["condition"].fillna("Unknown")

missing_after = df["condition"].isnull().sum()
unknown_count = (df["condition"] == "Unknown").sum()

print(f"Missing condition values before: {missing_before}")
print(f"Missing condition values after: {missing_after}")
print(f"Number of 'Unknown' condition values: {unknown_count}")

In [ ]:
# Convert HTML entities into valid text
df["review_clean"] = df["review"].astype(str).apply(html.unescape)

html_pattern = r"&[a-zA-Z]+;|&#\d+;"

# Sanity check: compare number of HTML entities before and after cleaning
html_count_before = df["review"].astype(str).str.contains(html_pattern, regex=True, na=False).sum()
html_count_after = df["review_clean"].astype(str).str.contains(html_pattern, regex=True, na=False).sum()

print(f"Reviews with HTML entities before cleaning: {html_count_before:,}")
print(f"Reviews with HTML entities after cleaning: {html_count_after:,}")

# Display a few examples where the text changed
changed_examples = df[df["review"].astype(str) != df["review_clean"].astype(str)].head(3)

print(f"Number of reviews changed after HTML unescaping: {len(df[df['review'].astype(str) != df['review_clean'].astype(str)]):,}")

for idx, row in changed_examples.iterrows():
    print("Original review:")
    print(row["review"])
    print("Cleaned review:")
    print(row["review_clean"])
    print()

In [ ]:
# Normalize whitespace in the cleaned review text, remove duplicated spaces, tabs, and line breaks
def normalize_spaces(text):
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["review_clean"] = df["review_clean"].apply(normalize_spaces)

# Sanity check: make sure there are no empty cleaned reviews
empty_reviews = (df["review_clean"].str.len() == 0).sum()

print(f"Number of empty cleaned reviews: {empty_reviews}")

# Display a few cleaned reviews
display(df[["review", "review_clean"]].sample(3, random_state=42))

In [ ]:
# Create two possible input formats for the model:
# 1. review-only: uses only the cleaned patient review
# 2. metadata-enriched: combines drug name, condition, and review

df["model_text_review_only"] = df["review_clean"]

df["model_text_with_metadata"] = (
    "Drug: " + df["drugName"].astype(str) +
    " | Condition: " + df["condition"].astype(str) +
    " | Review: " + df["review_clean"].astype(str) +
    " | usefulCount: " + df["usefulCount"].astype(str)
)

# Sanity checks
print("Example - review only:")
print(df["model_text_review_only"].iloc[0])
print()

print("Example - with metadata:")
print(df["model_text_with_metadata"].iloc[0])

In [ ]:
# Encode the original rating values as labels from 0 to 9
# Original rating: 1-10
# Encoded label: 0-9

df["label_10class"] = df["rating"].astype(int) - 1

# Sanity checks
print("Original rating values:")
print(sorted(df["rating"].unique()))

print("\nEncoded label values:")
print(sorted(df["label_10class"].unique()))

In [ ]:
# Create a 3-class sentiment label using the same mapping as the reference paper:
# ratings 1-4   -> negative
# ratings 5-6   -> neutral
# ratings 7-10  -> positive
# I created both a textual label and a numeric label

def rating_to_sentiment_text(r):
    if r <= 4:
        return "negative"
    elif r < 7:
        return "neutral"
    return "positive"

def rating_to_sentiment_label(r):
    if r <= 4:
        return 0   # negative
    elif r < 7:
        return 1   # neutral
    return 2       # positive

df["sentiment_3class"] = df["rating"].apply(rating_to_sentiment_text)
df["label_3class"] = df["rating"].apply(rating_to_sentiment_label)

# Sanity checks
print("3-class sentiment distribution:")
display(df["sentiment_3class"].value_counts(normalize=True).sort_index() * 100)

print("Numeric 3-class labels:")
print(sorted(df["label_3class"].unique()))

In [ ]:
# Create a log-transformed version of usefulCount
# Because of the long-tail distribution, log(1 + usefulCount) reduces the impact of outliers
# This feature can be used later in models that combine text and numeric metadata

df["log_usefulCount"] = np.log1p(df["usefulCount"])

# Sanity checks
display(df[["usefulCount", "log_usefulCount"]].describe())

In [ ]:
# Check the final columns created during preprocessing before splitting the data

expected_columns = [
    "condition",
    "review_clean",
    "model_text_review_only",
    "model_text_with_metadata",
    "label_10class",
    "sentiment_3class",
    "label_3class",
    "log_usefulCount"
]

for col in expected_columns:
    print(f"{col}: {'Exists' if col in df.columns else 'Missing'}")

In [ ]:
# Create a stratified train/validation split from the training set only

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label_10class"]
)

print(f"Train split shape: {train_df.shape}")
print(f"Validation split shape: {val_df.shape}")

# Sanity checks
print(f"Total after split: {len(train_df) + len(val_df)}")
print(f"Original df size: {len(df)}")

In [ ]:
# Save the preprocessed train and validation splits

train_df.to_csv("preprocessed_train.csv", index=False)
val_df.to_csv("preprocessed_val.csv", index=False)

print("Saved files: preprocessed_train.csv and preprocessed_val.csv")

## Modeling

In [ ]:
# Load preprocessed data
train_df = pd.read_csv("preprocessed_train.csv")
val_df = pd.read_csv("preprocessed_val.csv")

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)

# Main text column for the first modeling round (switch to "model_text_with_metadata" later to compare metadata-enriched input)
TEXT_COL = "model_text_review_only"

# Target column for 10-class rating prediction
LABEL_COL = "label_10class"

NUM_CLASSES = 10

# Optional: use a sample for fast debugging
# Set USE_SAMPLE = False for final runs
USE_SAMPLE = False
SAMPLE_SIZE = 20000

if USE_SAMPLE:
    train_df = train_df.sample(min(SAMPLE_SIZE, len(train_df)), random_state=SEED)
    val_df = val_df.sample(min(5000, len(val_df)), random_state=SEED)

print("Text column:", TEXT_COL)
print("Label column:", LABEL_COL)

In [ ]:
# Automatic progress saving and loading for model results and helpers for all models

PROGRESS_DIR = Path("modeling_progress")
PROGRESS_DIR.mkdir(parents=True, exist_ok=True)

# Initialize global containers only if they do not already exist to prevent accidental deletion if the cell is re-run
if "model_results" not in globals():
    model_results = []

if "confusion_matrices" not in globals():
    confusion_matrices = {}

if "training_histories" not in globals():
    training_histories = {}

if "training_logs" not in globals():
    training_logs = {}
if "interpretability_outputs" not in globals():
    interpretability_outputs = {}


def make_json_serializable(obj):
    """
    Convert common numpy/pandas objects into JSON-serializable objects.
    """
    if isinstance(obj, dict):
        return {str(k): make_json_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_serializable(v) for v in obj]
    if isinstance(obj, tuple):
        return [make_json_serializable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, pd.DataFrame):
        return obj.to_dict(orient="records")
    if isinstance(obj, pd.Series):
        return obj.to_dict()
    return obj


def save_modeling_progress(save_dir=PROGRESS_DIR):
    """
    Save all modeling progress collected so far.

    Saved files:
    - model_results.csv
    - model_results.json
    - confusion_matrices.pkl
    - training_histories.pkl
    - training_logs.pkl
    - interpretability_outputs.pkl
    - interpretability CSV files
    - modeling_progress_summary.json
    """
    save_path = Path(save_dir)
    save_path.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Save model metrics
    results_df = pd.DataFrame(model_results)

    if len(results_df) > 0:
        results_df.to_csv(save_path / "model_results.csv", index=False)

        with open(save_path / "model_results.json", "w", encoding="utf-8") as f:
            json.dump(
                make_json_serializable(model_results),
                f,
                indent=2,
                ensure_ascii=False
            )

    # Save confusion matrices
    with open(save_path / "confusion_matrices.pkl", "wb") as f:
        pickle.dump(confusion_matrices, f)

    # Save PyTorch training histories
    with open(save_path / "training_histories.pkl", "wb") as f:
        pickle.dump(training_histories, f)

    # Save Hugging Face training logs
    with open(save_path / "training_logs.pkl", "wb") as f:
        pickle.dump(training_logs, f)

    # Save interpretability outputs, for example Logistic Regression top features
    with open(save_path / "interpretability_outputs.pkl", "wb") as f:
        pickle.dump(interpretability_outputs, f)

    # Also save every interpretability DataFrame as a readable CSV
    interpretability_dir = save_path / "interpretability_outputs"
    interpretability_dir.mkdir(parents=True, exist_ok=True)

    for output_name, output_value in interpretability_outputs.items():
        safe_name = output_name.replace(" ", "_").replace("/", "_").replace("+", "plus")

        if isinstance(output_value, pd.DataFrame):
            output_value.to_csv(interpretability_dir / f"{safe_name}.csv", index=False)

        elif isinstance(output_value, dict):
            for inner_name, inner_value in output_value.items():
                inner_safe_name = str(inner_name).replace(" ", "_").replace("/", "_").replace("+", "plus")

                if isinstance(inner_value, pd.DataFrame):
                    inner_value.to_csv(
                        interpretability_dir / f"{safe_name}_{inner_safe_name}.csv",
                        index=False
                    )

    summary = {
        "timestamp": timestamp,
        "save_dir": str(save_path.resolve()),
        "num_completed_models": len(model_results),
        "completed_models": [r["model"] for r in model_results],
        "num_confusion_matrices": len(confusion_matrices),
        "available_training_histories": list(training_histories.keys()),
        "available_training_logs": list(training_logs.keys()),
        "available_interpretability_outputs": list(interpretability_outputs.keys())
    }

    with open(save_path / "modeling_progress_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

    print(f"Progress saved automatically to: {save_path.resolve()}")

    return summary


def load_modeling_progress(save_dir=PROGRESS_DIR, restore_to_globals=True):
    """
    Load previously saved modeling progress.

    If restore_to_globals=True, the loaded results are restored into:
    - model_results
    - confusion_matrices
    - training_histories
    - training_logs
    - interpretability_outputs
    """
    save_path = Path(save_dir)

    loaded_model_results = []
    loaded_confusion_matrices = {}
    loaded_training_histories = {}
    loaded_training_logs = {}
    loaded_interpretability_outputs = {}

    # Load metrics
    results_file = save_path / "model_results.json"
    if results_file.exists():
        with open(results_file, "r", encoding="utf-8") as f:
            loaded_model_results = json.load(f)
    else:
        print("No saved model_results.json found.")

    # Load confusion matrices
    cm_file = save_path / "confusion_matrices.pkl"
    if cm_file.exists():
        with open(cm_file, "rb") as f:
            loaded_confusion_matrices = pickle.load(f)
    else:
        print("No saved confusion_matrices.pkl found.")

    # Load PyTorch histories
    histories_file = save_path / "training_histories.pkl"
    if histories_file.exists():
        with open(histories_file, "rb") as f:
            loaded_training_histories = pickle.load(f)
    else:
        print("No saved training_histories.pkl found.")

    # Load Hugging Face logs
    logs_file = save_path / "training_logs.pkl"
    if logs_file.exists():
        with open(logs_file, "rb") as f:
            loaded_training_logs = pickle.load(f)
    else:
        print("No saved training_logs.pkl found.")

    # Load interpretability outputs
    interpretability_file = save_path / "interpretability_outputs.pkl"
    if interpretability_file.exists():
        with open(interpretability_file, "rb") as f:
            loaded_interpretability_outputs = pickle.load(f)
    else:
        print("No saved interpretability_outputs.pkl found.")

    if restore_to_globals:
        globals()["model_results"] = loaded_model_results
        globals()["confusion_matrices"] = loaded_confusion_matrices
        globals()["training_histories"] = loaded_training_histories
        globals()["training_logs"] = loaded_training_logs
        globals()["interpretability_outputs"] = loaded_interpretability_outputs

    loaded_results_df = pd.DataFrame(loaded_model_results)

    print("Progress loaded.")
    print("Completed models:", loaded_results_df["model"].tolist() if "model" in loaded_results_df.columns else [])
    print("Interpretability outputs:", list(loaded_interpretability_outputs.keys()))

    if len(loaded_results_df) > 0:
        display(loaded_results_df)

    return (
        loaded_results_df,
        loaded_confusion_matrices,
        loaded_training_histories,
        loaded_training_logs,
        loaded_interpretability_outputs
    )


def evaluate_model(model_name, y_true_labels, y_pred_labels):
    """
    Evaluate a model and automatically save the results.

    This function:
    1. Calculates all metrics.
    2. Saves the confusion matrix.
    3. Updates model_results.
    4. Prints a classification report.
    5. Automatically saves progress to disk.
    """
    y_true_labels = np.array(y_true_labels)
    y_pred_labels = np.array(y_pred_labels)

    y_true_ratings = y_true_labels + 1
    y_pred_ratings = y_pred_labels + 1

    acc = accuracy_score(y_true_labels, y_pred_labels)
    macro_precision = precision_score(y_true_labels, y_pred_labels, average="macro", zero_division=0)
    macro_recall = recall_score(y_true_labels, y_pred_labels, average="macro", zero_division=0)
    macro_f1 = f1_score(y_true_labels, y_pred_labels, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_true_labels, y_pred_labels, average="weighted", zero_division=0)
    mae = mean_absolute_error(y_true_ratings, y_pred_ratings)

    cm = confusion_matrix(
        y_true_labels,
        y_pred_labels,
        labels=list(range(10))
    )

    confusion_matrices[model_name] = cm

    result = {
        "model": model_name,
        "accuracy": acc,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "mae": mae
    }

    # Avoid duplicate rows if the same model is evaluated again.
    global model_results
    model_results = [r for r in model_results if r.get("model") != model_name]
    model_results.append(result)

    print(f"Results for {model_name}")
    print("=" * 80)

    for key, value in result.items():
        if key != "model":
            print(f"{key}: {value:.4f}")

    print("\nClassification report:")
    print(
        classification_report(
            y_true_labels,
            y_pred_labels,
            labels=list(range(10)),
            target_names=[str(i) for i in range(1, 11)],
            zero_division=0
        )
    )

    # Automatic save after every completed model evaluation.
    save_modeling_progress()

    return result


def plot_confusion_matrix(model_name):
    """
    Plot the confusion matrix for a completed model.
    """
    if model_name not in confusion_matrices:
        print(f"No confusion matrix found for model: {model_name}")
        return

    cm = confusion_matrices[model_name]

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=list(range(1, 11)),
        yticklabels=list(range(1, 11))
    )
    plt.title(f"Confusion Matrix - {model_name}")
    plt.xlabel("Predicted Rating")
    plt.ylabel("True Rating")
    plt.show()


def show_results_table(sort_by="macro_f1"):
    """
    Display the current results table.
    """
    results_df = pd.DataFrame(model_results)

    if len(results_df) == 0:
        print("No model results yet.")
        return None

    if sort_by in results_df.columns:
        results_df = results_df.sort_values(by=sort_by, ascending=False)

    display(results_df)

    return results_df

### Model 1: Majority Class Baseline

In [ ]:
# This baseline always predicts the most frequent class in the training data.
# It is important because the EDA showed that the rating distribution is imbalanced.
# Any other model should perform better than this baseline.

X_train = train_df[[TEXT_COL]]
y_train = train_df[LABEL_COL]

X_val = val_df[[TEXT_COL]]
y_val = val_df[LABEL_COL]

majority_baseline = DummyClassifier(strategy="most_frequent", random_state=SEED)
majority_baseline.fit(X_train, y_train)

majority_preds = majority_baseline.predict(X_val)

evaluate_model("Majority Class Baseline", y_val, majority_preds)
plot_confusion_matrix("Majority Class Baseline")

### Model 2: TF-IDF + Logistic Regression

In [ ]:
# This is a strong classical machine learning baseline.
# It is also useful for comparison with the reference paper, which used n-gram based lexical features and logistic regression.

tfidf_vectorizer = TfidfVectorizer(
    max_features=100000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = tfidf_vectorizer.fit_transform(train_df[TEXT_COL].astype(str))
X_val_tfidf = tfidf_vectorizer.transform(val_df[TEXT_COL].astype(str))

log_reg_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    solver="saga",
    n_jobs=-1,
    random_state=SEED
)

log_reg_model.fit(X_train_tfidf, train_df[LABEL_COL])
log_reg_preds = log_reg_model.predict(X_val_tfidf)

evaluate_model("TF-IDF + Logistic Regression", val_df[LABEL_COL], log_reg_preds)
plot_confusion_matrix("TF-IDF + Logistic Regression")

In [ ]:
# Interpretability for Logistic Regression.
# Inspect the most influential TF-IDF features for selected rating classes.

feature_names = np.array(tfidf_vectorizer.get_feature_names_out())

def show_top_features_for_class(class_label, top_n=20, save_output=True):
    """
    Show and optionally save the top TF-IDF Logistic Regression features
    for a specific predicted rating class.

    class_label is zero-based:
    - 0 means rating 1
    - 9 means rating 10
    """
    feature_names = np.array(tfidf_vectorizer.get_feature_names_out())
    coefs = log_reg_model.coef_[class_label]

    top_indices = np.argsort(coefs)[-top_n:][::-1]

    top_features = pd.DataFrame({
        "predicted_rating": class_label + 1,
        "feature": feature_names[top_indices],
        "coefficient": coefs[top_indices]
    })

    print(f"Top features for predicted rating {class_label + 1}:")
    display(top_features)

    if save_output:
        output_key = f"Logistic Regression Top Features Rating {class_label + 1}"
        interpretability_outputs[output_key] = top_features

        save_modeling_progress()

    return top_features


def save_all_logistic_regression_top_features(top_n=20):
    """
    Save top Logistic Regression features for all 10 rating classes.
    """
    all_top_features = {}

    for class_label in range(NUM_CLASSES):
        top_features = show_top_features_for_class(
            class_label=class_label,
            top_n=top_n,
            save_output=False
        )

        all_top_features[f"rating_{class_label + 1}"] = top_features

    interpretability_outputs["Logistic Regression Top Features All Ratings"] = all_top_features

    save_modeling_progress()

    print("All Logistic Regression top features were saved.")

    return all_top_features


show_top_features_for_class(0, top_n=10)
show_top_features_for_class(9, top_n=10)

logreg_top_features = save_all_logistic_regression_top_features(top_n=20)

### Model 3: TextCNN

In [ ]:
# TextCNN and BiLSTM do not use pretrained transformer tokenizers.
# Therefore, build a simple word-level vocabulary from the training text only.
# This avoids data leakage and keeps the validation set unseen during vocabulary construction.

MAX_WORDS = 50000
MAX_LEN = 256
EMBEDDING_DIM = 128
NUM_CLASSES = 10
BATCH_SIZE = 128

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


def simple_tokenize(text):
    """
    Simple tokenizer for classical neural text models.
    
    The function lowercases the text and extracts word-like tokens.
    This is intentionally simple because TextCNN/BiLSTM are used as deep learning baselines,
    while transformer models will later use their own pretrained tokenizers.
    """
    text = str(text).lower()
    tokens = re.findall(r"\b\w+\b", text)
    return tokens


# Build vocabulary from train text only
word_counter = Counter()

for text in train_df[TEXT_COL].astype(str):
    word_counter.update(simple_tokenize(text))

# Reserve special tokens:
# 0 = PAD, used for padding shorter reviews
# 1 = OOV, used for words not seen in the training vocabulary
vocab = {
    "<PAD>": 0,
    "<OOV>": 1
}

# Add the most common words to the vocabulary
for word, count in word_counter.most_common(MAX_WORDS - 2):
    vocab[word] = len(vocab)

vocab_size = len(vocab)

print(f"Vocabulary size: {vocab_size:,}")


def encode_text(text, vocab, max_len=MAX_LEN):
    """
    Convert a review text into a fixed-length list of token ids.
    
    Steps:
    1. Tokenize the text.
    2. Map each token to its integer id.
    3. Truncate long reviews to max_len.
    4. Pad short reviews with 0 until max_len.
    """
    tokens = simple_tokenize(text)
    token_ids = [vocab.get(token, vocab["<OOV>"]) for token in tokens]
    
    # Truncate
    token_ids = token_ids[:max_len]
    
    # Pad
    if len(token_ids) < max_len:
        token_ids += [vocab["<PAD>"]] * (max_len - len(token_ids))
    
    return token_ids


class DrugReviewTextDataset(TorchDataset):
    """
    PyTorch Dataset for TextCNN and BiLSTM.
    
    Each item contains:
    - input_ids: fixed-length tensor of token ids
    - label: encoded rating label from 0 to 9
    """
    
    def __init__(self, texts, labels, vocab, max_len=MAX_LEN):
        self.texts = texts.astype(str).tolist()
        self.labels = labels.astype(int).tolist()
        self.vocab = vocab
        self.max_len = max_len
        
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        input_ids = encode_text(
            self.texts[idx],
            vocab=self.vocab,
            max_len=self.max_len
        )
        
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }


train_torch_dataset = DrugReviewTextDataset(
    texts=train_df[TEXT_COL],
    labels=train_df[LABEL_COL],
    vocab=vocab,
    max_len=MAX_LEN
)

val_torch_dataset = DrugReviewTextDataset(
    texts=val_df[TEXT_COL],
    labels=val_df[LABEL_COL],
    vocab=vocab,
    max_len=MAX_LEN
)

train_loader = DataLoader(
    train_torch_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_torch_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Sanity checks
first_batch = next(iter(train_loader))

print("input_ids batch shape:", first_batch["input_ids"].shape)
print("label batch shape:", first_batch["label"].shape)
print("Minimum label:", first_batch["label"].min().item())
print("Maximum label:", first_batch["label"].max().item())

assert first_batch["input_ids"].shape[1] == MAX_LEN, "Sequence length does not match MAX_LEN."
assert first_batch["label"].min().item() >= 0, "Labels should start from 0."
assert first_batch["label"].max().item() <= 9, "Labels should end at 9."

In [ ]:
# Shared PyTorch training and prediction functions for both TextCNN and BiLSTM to keep the experiments comparable.

def train_torch_model(
    model,
    train_loader,
    val_loader,
    model_name,
    num_epochs=5,
    learning_rate=1e-3,
    use_class_weights=True
):
    """
    Train a PyTorch text classification model.

    At the end of training, the training history is saved in training_histories
    and progress is automatically saved to disk.
    """
    model = model.to(device)

    if use_class_weights:
        class_weights = compute_class_weight(
            class_weight="balanced",
            classes=np.arange(NUM_CLASSES),
            y=train_df[LABEL_COL].values
        )
        class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    else:
        criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    best_val_loss = float("inf")
    best_model_state = None

    history = {
        "train_loss": [],
        "val_loss": []
    }

    for epoch in range(num_epochs):
        model.train()
        total_train_loss = 0

        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad()

            logits = model(input_ids)
            loss = criterion(logits, labels)

            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_loader)

        model.eval()
        total_val_loss = 0

        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch["input_ids"].to(device)
                labels = batch["label"].to(device)

                logits = model(input_ids)
                loss = criterion(logits, labels)

                total_val_loss += loss.item()

        avg_val_loss = total_val_loss / len(val_loader)

        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(avg_val_loss)

        print(
            f"{model_name} | Epoch {epoch + 1}/{num_epochs} | "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Val Loss: {avg_val_loss:.4f}"
        )

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = model.state_dict()

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    # Save training history automatically.
    training_histories[model_name] = history
    save_modeling_progress()

    return model, history


def predict_torch_model(model, data_loader):
    """
    Generate predictions for a PyTorch classification model.
    """
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["label"].to(device)

            logits = model(input_ids)
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return np.array(all_labels), np.array(all_preds)


def plot_training_history(history, model_name):
    """
    Plot train and validation loss over epochs.
    """
    plt.figure(figsize=(8, 5))
    plt.plot(history["train_loss"], label="Train Loss")
    plt.plot(history["val_loss"], label="Validation Loss")
    plt.title(f"Training History - {model_name}")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

In [ ]:
# TextCNN is useful for capturing local phrase patterns.
# For example, expressions such as "side effects" and "highly recommend" can be detected using convolution filters over word embeddings.

class TextCNN(nn.Module):
    """
    TextCNN model for 10-class rating prediction.
    
    Architecture:
    1. Word Embedding layer
    2. Several Conv1D layers with different kernel sizes
    3. Global max pooling
    4. Dropout
    5. Linear classification layer
    """
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        num_classes,
        kernel_sizes=[3, 4, 5],
        num_filters=128,
        dropout=0.5
    ):
        super(TextCNN, self).__init__()
        
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )
        
        self.convs = nn.ModuleList([
            nn.Conv1d(
                in_channels=embedding_dim,
                out_channels=num_filters,
                kernel_size=k
            )
            for k in kernel_sizes
        ])
        
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), num_classes)
    
    def forward(self, input_ids):
        # input_ids shape: [batch_size, seq_len]
        
        x = self.embedding(input_ids)
        # x shape: [batch_size, seq_len, embedding_dim]
        
        x = x.permute(0, 2, 1)
        # x shape: [batch_size, embedding_dim, seq_len]
        
        conv_outputs = [
            torch.relu(conv(x))
            for conv in self.convs
        ]
        
        pooled_outputs = [
            torch.max(conv_out, dim=2).values
            for conv_out in conv_outputs
        ]
        
        x = torch.cat(pooled_outputs, dim=1)
        x = self.dropout(x)
        
        logits = self.fc(x)
        
        return logits


textcnn_model = TextCNN(
    vocab_size=vocab_size,
    embedding_dim=EMBEDDING_DIM,
    num_classes=NUM_CLASSES
)

print(textcnn_model)

In [ ]:
# Train and evaluate the PyTorch TextCNN model

textcnn_model, textcnn_history = train_torch_model(
    model=textcnn_model,
    train_loader=train_loader,
    val_loader=val_loader,
    model_name="TextCNN",
    num_epochs=5,
    learning_rate=1e-3,
    use_class_weights=True
)

plot_training_history(textcnn_history, "TextCNN")

y_true_textcnn, y_pred_textcnn = predict_torch_model(
    model=textcnn_model,
    data_loader=val_loader
)

evaluate_model("TextCNN", y_true_textcnn, y_pred_textcnn)
plot_confusion_matrix("TextCNN")

### Model 4: BiLSTM

In [ ]:
# BiLSTM reads the review from both directions and captures sequential context.
# This is useful for sentences where meaning depends on word order such as "did not work", "no side effects".

class BiLSTMClassifier(nn.Module):
    """
    BiLSTM model for 10-class rating prediction.
    
    Architecture:
    1. Word Embedding layer
    2. Bidirectional LSTM
    3. Global max pooling over sequence outputs
    4. Dropout
    5. Linear classification layer
    """
    
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        num_classes,
        dropout=0.5
    ):
        super(BiLSTMClassifier, self).__init__()
        
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )
        
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )
        
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
    
    def forward(self, input_ids):
        # input_ids shape: [batch_size, seq_len]
        
        x = self.embedding(input_ids)
        # x shape: [batch_size, seq_len, embedding_dim]
        
        lstm_out, _ = self.lstm(x)
        # lstm_out shape: [batch_size, seq_len, hidden_dim * 2]
        
        pooled = torch.max(lstm_out, dim=1).values
        # pooled shape: [batch_size, hidden_dim * 2]
        
        pooled = self.dropout(pooled)
        
        logits = self.fc(pooled)
        
        return logits


bilstm_model = BiLSTMClassifier(
    vocab_size=vocab_size,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=128,
    num_classes=NUM_CLASSES
)

print(bilstm_model)

In [ ]:
# Train and evaluate the PyTorch BiLSTM model

bilstm_model, bilstm_history = train_torch_model(
    model=bilstm_model,
    train_loader=train_loader,
    val_loader=val_loader,
    model_name="BiLSTM",
    num_epochs=5,
    learning_rate=1e-3,
    use_class_weights=True
)

plot_training_history(bilstm_history, "BiLSTM")

y_true_bilstm, y_pred_bilstm = predict_torch_model(
    model=bilstm_model,
    data_loader=val_loader
)

evaluate_model("BiLSTM", y_true_bilstm, y_pred_bilstm)
plot_confusion_matrix("BiLSTM")

### Model 5: DistilBERT Fine-Tuning

In [ ]:
# Helper functions and Dataset class for transformer-based models

class DrugReviewTorchDataset(TorchDataset):
    """
    PyTorch Dataset for transformer-based models.
    """

    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts.astype(str).tolist()
        self.labels = labels.astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding=False
        )

        item = {
            key: torch.tensor(value, dtype=torch.long)
            for key, value in encoding.items()
        }

        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)

        return item


def compute_metrics_for_trainer(eval_pred):
    """
    Metrics for Hugging Face Trainer.
    """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_precision": precision_score(labels, preds, average="macro", zero_division=0),
        "macro_recall": recall_score(labels, preds, average="macro", zero_division=0),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels, preds, average="weighted", zero_division=0),
        "mae": mean_absolute_error(labels + 1, preds + 1)
    }


def train_transformer_model(
    model_checkpoint,
    model_name,
    train_df,
    val_df,
    text_col=TEXT_COL,
    label_col=LABEL_COL,
    max_length=256,
    num_train_epochs=2,
    train_batch_size=16,
    eval_batch_size=32,
    learning_rate=2e-5,
    gradient_accumulation_steps=1
):
    """
    Train and evaluate a Hugging Face transformer model.

    The Trainer logs and evaluation results are automatically saved.
    """
    print(f"Training model: {model_name}")
    print(f"Model checkpoint: {model_checkpoint}")
    print(f"Train samples: {len(train_df):,}")
    print(f"Validation samples: {len(val_df):,}")
    print(f"Text column: {text_col}")
    print(f"Label column: {label_col}")

    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

    train_dataset = DrugReviewTorchDataset(
        texts=train_df[text_col],
        labels=train_df[label_col],
        tokenizer=tokenizer,
        max_length=max_length
    )

    val_dataset = DrugReviewTorchDataset(
        texts=val_df[text_col],
        labels=val_df[label_col],
        tokenizer=tokenizer,
        max_length=max_length
    )

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_checkpoint,
        num_labels=NUM_CLASSES,
        use_safetensors=True
    )

    output_dir = f"./{model_name.replace(' ', '_').replace('/', '_')}"

    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=learning_rate,
        per_device_train_batch_size=train_batch_size,
        per_device_eval_batch_size=eval_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        num_train_epochs=num_train_epochs,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        logging_steps=100,
        save_total_limit=1,
        report_to="none",
        seed=SEED,
        fp16=torch.cuda.is_available()
    )

    trainer_kwargs = {
        "model": model,
        "args": training_args,
        "train_dataset": train_dataset,
        "eval_dataset": val_dataset,
        "data_collator": data_collator,
        "compute_metrics": compute_metrics_for_trainer
    }

    # Newer versions of Transformers use processing_class.
    # Older versions use tokenizer.
    try:
        trainer = Trainer(
            **trainer_kwargs,
            processing_class=tokenizer
        )
    except TypeError:
        trainer = Trainer(
            **trainer_kwargs,
            tokenizer=tokenizer
        )

    trainer.train()

    # Save Hugging Face log history after training.
    training_logs[model_name] = make_json_serializable(trainer.state.log_history)
    save_modeling_progress()

    predictions = trainer.predict(val_dataset)
    y_pred = np.argmax(predictions.predictions, axis=1)
    y_true = predictions.label_ids

    evaluate_model(model_name, y_true, y_pred)
    plot_confusion_matrix(model_name)

    # Save logs again after prediction/evaluation.
    training_logs[model_name] = make_json_serializable(trainer.state.log_history)
    save_modeling_progress()

    return trainer, tokenizer, model

In [ ]:
# DistilBERT is a smaller and faster version of BERT.

distilbert_trainer, distilbert_tokenizer, distilbert_model = train_transformer_model(
    model_checkpoint="distilbert-base-uncased",
    model_name="DistilBERT",
    train_df=train_df,
    val_df=val_df,
    text_col=TEXT_COL,
    label_col=LABEL_COL,
    max_length=256,
    num_train_epochs=2,
    train_batch_size=16,
    eval_batch_size=32,
    learning_rate=2e-5
)

### Model 6: BERT / RoBERTa Fine-Tuning

In [ ]:
# BERT is a contextual transformer model that can capture negation, sentence structure, and semantic meaning.

bert_trainer, bert_tokenizer, bert_model = train_transformer_model(
    model_checkpoint="bert-base-uncased",
    model_name="BERT Base",
    train_df=train_df,
    val_df=val_df,
    text_col=TEXT_COL,
    label_col=LABEL_COL,
    max_length=256,
    num_train_epochs=2,
    train_batch_size=16,
    eval_batch_size=32,
    learning_rate=2e-5
)

In [ ]:
# RoBERTa is a stronger BERT-like model trained with improved pretraining strategy.

roberta_trainer, roberta_tokenizer, roberta_model = train_transformer_model(
    model_checkpoint="roberta-base",
    model_name="RoBERTa Base",
    train_df=train_df,
    val_df=val_df,
    text_col=TEXT_COL,
    label_col=LABEL_COL,
    max_length=256,
    num_train_epochs=2,
    train_batch_size=16,
    eval_batch_size=32,
    learning_rate=2e-5
)

### Model 7: Medical-Domain Transformer - BioClinicalBERT, PubMedBERT

In [ ]:
# BioClinicalBERT is a medical-domain transformer model.
# It may be useful because the reviews include medical conditions, treatments, and drug-related language.

bioclinicalbert_trainer, bioclinicalbert_tokenizer, bioclinicalbert_model = train_transformer_model(
    model_checkpoint="emilyalsentzer/Bio_ClinicalBERT",
    model_name="BioClinicalBERT",
    train_df=train_df,
    val_df=val_df,
    text_col=TEXT_COL,
    label_col=LABEL_COL,
    max_length=256,
    num_train_epochs=2,
    train_batch_size=16,
    eval_batch_size=32,
    learning_rate=2e-5
)

In [ ]:
# Model 7B: PubMedBERT
# PubMedBERT is pretrained on biomedical literature.
# This allows us to test whether biomedical-domain pretraining improves rating prediction on drug reviews.

pubmedbert_trainer, pubmedbert_tokenizer, pubmedbert_model = train_transformer_model(
    model_checkpoint="microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext",
    model_name="PubMedBERT",
    train_df=train_df,
    val_df=val_df,
    text_col=TEXT_COL,
    label_col=LABEL_COL,
    max_length=256,
    num_train_epochs=2,
    train_batch_size=16,
    eval_batch_size=32,
    learning_rate=2e-5
)

## Additional Transformer Experiments with Metadata-Enriched Input

The initial experiments used only the review text as input.  
However, the TF-IDF baseline achieved the strongest macro-F1, possibly because sparse n-gram features capture strong lexical signals and class-specific phrases.

To perform a small and focused improvement round, I rerun the strongest transformer models using a metadata-enriched input format:

Drug: [drug name] | Condition: [condition] | Review: [review text]

This allows the transformer models to use additional contextual information about the medication and the treated condition, while keeping the modeling approach fully deep-learning based.

In [ ]:
# Load previously saved results before running additional experiments

loaded_results_df, loaded_confusion_matrices, loaded_training_histories, loaded_training_logs, loaded_interpretability_outputs = load_modeling_progress(
    save_dir=PROGRESS_DIR,
    restore_to_globals=True
)

# Ensure that the metadata-enriched input column exists.
# This input gives the model access to the drug name and medical condition in addition to the review text.

if "model_text_with_metadata" not in train_df.columns:
    train_df["model_text_with_metadata"] = (
        "Drug: " + train_df["drugName"].astype(str) +
        " | Condition: " + train_df["condition"].astype(str) +
        " | Review: " + train_df["review_clean"].astype(str)
    )

if "model_text_with_metadata" not in val_df.columns:
    val_df["model_text_with_metadata"] = (
        "Drug: " + val_df["drugName"].astype(str) +
        " | Condition: " + val_df["condition"].astype(str) +
        " | Review: " + val_df["review_clean"].astype(str)
    )

print("Metadata-enriched input examples:")
display(train_df[["model_text_review_only", "model_text_with_metadata", LABEL_COL]].head(3))

In [ ]:
# Helper function to avoid rerunning an experiment that already exists in the saved results

def model_already_completed(model_name):
    completed_models = [result["model"] for result in model_results]
    return model_name in completed_models


def print_completed_models():
    completed_models = [result["model"] for result in model_results]
    print("Completed models:")
    for model in completed_models:
        print("-", model)

print_completed_models()

In [ ]:
# Additional experiment: RoBERTa with metadata-enriched input.
# RoBERTa was one of the strongest deep learning models in terms of MAE.
# Test whether adding drug and condition information improves its ability to distinguish between similar reviews with different medical contexts.

if not model_already_completed("RoBERTa Base + Metadata"):
    roberta_metadata_trainer, roberta_metadata_tokenizer, roberta_metadata_model = train_transformer_model(
        model_checkpoint="roberta-base",
        model_name="RoBERTa Base + Metadata",
        train_df=train_df,
        val_df=val_df,
        text_col="model_text_with_metadata",
        label_col=LABEL_COL,
        max_length=256,
        num_train_epochs=2,
        train_batch_size=16,
        eval_batch_size=32,
        learning_rate=2e-5
    )
else:
    print("RoBERTa Base + Metadata already completed. Skipping.")

In [ ]:
# Additional experiment 2: BERT with metadata-enriched input.
# BERT Base achieved competitive results among the transformer models.
# This experiment checks whether explicit drug and condition information helps the model improve rating prediction compared to review-only input.

if not model_already_completed("BERT Base + Metadata"):
    bert_metadata_trainer, bert_metadata_tokenizer, bert_metadata_model = train_transformer_model(
        model_checkpoint="bert-base-uncased",
        model_name="BERT Base + Metadata",
        train_df=train_df,
        val_df=val_df,
        text_col="model_text_with_metadata",
        label_col=LABEL_COL,
        max_length=256,
        num_train_epochs=2,
        train_batch_size=16,
        eval_batch_size=32,
        learning_rate=2e-5
    )
else:
    print("BERT Base + Metadata already completed. Skipping.")

In [ ]:
# Additional experiment 3: BioClinicalBERT with metadata-enriched input.
# BioClinicalBERT is pretrained on clinical text, so it may benefit from explicit medical context such as the condition and the drug name.
# This experiment tests whether domain-specific pretraining becomes more useful when the medical metadata is included in the input.

if not model_already_completed("BioClinicalBERT + Metadata"):
    bioclinicalbert_metadata_trainer, bioclinicalbert_metadata_tokenizer, bioclinicalbert_metadata_model = train_transformer_model(
        model_checkpoint="emilyalsentzer/Bio_ClinicalBERT",
        model_name="BioClinicalBERT + Metadata",
        train_df=train_df,
        val_df=val_df,
        text_col="model_text_with_metadata",
        label_col=LABEL_COL,
        max_length=256,
        num_train_epochs=2,
        train_batch_size=16,
        eval_batch_size=32,
        learning_rate=2e-5
    )
else:
    print("BioClinicalBERT + Metadata already completed. Skipping.")

In [ ]:
# Save and reload all results after the metadata experiments to ensure that the final comparison table includes all models

save_modeling_progress()

loaded_results_df, loaded_confusion_matrices, loaded_training_histories, loaded_training_logs, loaded_interpretability_outputs = load_modeling_progress(
    save_dir=PROGRESS_DIR,
    restore_to_globals=True
)

## Additional Experiments: Ordinal-Aware Loss and 3-Class Sentiment Classification

The task was modeled as a 10-class classification problem, where each rating from 1 to 10 is treated as a separate class.

However, ratings are ordinal: predicting 8 instead of 9 is a smaller mistake than predicting 1 instead of 9. Standard cross-entropy loss does not explicitly account for this distance.

Therefore, I added two focused follow-up experiments:

1. Ordinal-aware loss for 10-class rating prediction.
2. 3-class sentiment classification using the mapping from the published paper:
   - ratings 1–4: negative
   - ratings 5–6: neutral
   - ratings 7–10: positive

These experiments are performed only on the strongest pretrained transformer models.

In [ ]:
# Load saved progress before adding new experiments

loaded_results_df, loaded_confusion_matrices, loaded_training_histories, loaded_training_logs, loaded_interpretability_outputs = load_modeling_progress(
    save_dir=PROGRESS_DIR,
    restore_to_globals=True
)

show_results_table()


def model_already_completed(model_name):
    """
    Check whether a model was already evaluated and saved.
    This prevents accidental reruns of long experiments.
    """
    completed_models = [result["model"] for result in model_results]
    return model_name in completed_models


def evaluate_model_flexible(
    model_name,
    y_true_labels,
    y_pred_labels,
    num_classes,
    target_names=None,
    task_name=None
):
    """
    Flexible evaluation function for both 10-class rating prediction
    and 3-class sentiment classification.

    It saves the results into the same global model_results table
    and also saves the confusion matrix.
    """
    y_true_labels = np.array(y_true_labels)
    y_pred_labels = np.array(y_pred_labels)

    acc = accuracy_score(y_true_labels, y_pred_labels)
    macro_precision = precision_score(y_true_labels, y_pred_labels, average="macro", zero_division=0)
    macro_recall = recall_score(y_true_labels, y_pred_labels, average="macro", zero_division=0)
    macro_f1 = f1_score(y_true_labels, y_pred_labels, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_true_labels, y_pred_labels, average="weighted", zero_division=0)
    mae = mean_absolute_error(y_true_labels, y_pred_labels)

    labels = list(range(num_classes))

    if target_names is None:
        target_names = [str(i) for i in labels]

    cm = confusion_matrix(
        y_true_labels,
        y_pred_labels,
        labels=labels
    )

    confusion_matrices[model_name] = cm

    result = {
        "model": model_name,
        "task": task_name if task_name is not None else f"{num_classes}-class",
        "accuracy": acc,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "mae": mae
    }

    # Avoid duplicate rows if the same model is evaluated again
    global model_results
    model_results = [r for r in model_results if r.get("model") != model_name]
    model_results.append(result)

    print(f"Results for {model_name}")
    print("=" * 80)

    for key, value in result.items():
        if key not in ["model", "task"]:
            print(f"{key}: {value:.4f}")

    print("\nClassification report:")
    print(
        classification_report(
            y_true_labels,
            y_pred_labels,
            labels=labels,
            target_names=target_names,
            zero_division=0
        )
    )

    save_modeling_progress()

    return result


def plot_confusion_matrix_flexible(model_name, class_labels):
    """
    Plot a confusion matrix for either 10-class or 3-class experiments.
    """
    if model_name not in confusion_matrices:
        print(f"No confusion matrix found for model: {model_name}")
        return

    cm = confusion_matrices[model_name]

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_labels,
        yticklabels=class_labels
    )
    plt.title(f"Confusion Matrix - {model_name}")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.show()

In [ ]:
# Shared dataset and helper functions for the additional transformer experiments

class FlexibleDrugReviewTorchDataset(TorchDataset):
    """
    PyTorch Dataset for transformer models.
    It supports both 10-class rating labels and 3-class sentiment labels.
    """

    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts.astype(str).tolist()
        self.labels = labels.astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding=False
        )

        item = {
            key: torch.tensor(value, dtype=torch.long)
            for key, value in encoding.items()
        }

        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)

        return item


def compute_class_weights_from_labels(labels, num_classes):
    """
    Compute balanced class weights from the training labels.
    These weights help reduce the effect of class imbalance.
    """
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(num_classes),
        y=np.array(labels)
    )

    return torch.tensor(class_weights, dtype=torch.float)


def make_compute_metrics(num_classes):
    """
    Create a metrics function for Hugging Face Trainer.
    """

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=1)

        return {
            "accuracy": accuracy_score(labels, preds),
            "macro_precision": precision_score(labels, preds, average="macro", zero_division=0),
            "macro_recall": recall_score(labels, preds, average="macro", zero_division=0),
            "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
            "weighted_f1": f1_score(labels, preds, average="weighted", zero_division=0),
            "mae": mean_absolute_error(labels, preds)
        }

    return compute_metrics

In [ ]:
# Ordinal-aware Trainer

class OrdinalAwareTrainer(Trainer):
    """
    Hugging Face Trainer with ordinal-aware loss.

    Loss = Weighted Cross Entropy + alpha * SmoothL1(expected_class, true_class)

    expected_class is computed from the predicted probability distribution:
    expected_class = sum_i P(class_i) * i
    """

    def __init__(self, *args, class_weights=None, alpha=0.5, num_classes=10, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.alpha = alpha
        self.num_classes = num_classes

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.logits

        if self.class_weights is not None:
            weights = self.class_weights.to(logits.device)
            ce_loss_fn = nn.CrossEntropyLoss(weight=weights)
        else:
            ce_loss_fn = nn.CrossEntropyLoss()

        ce_loss = ce_loss_fn(logits, labels)

        probs = torch.softmax(logits, dim=-1)

        class_values = torch.arange(
            self.num_classes,
            device=logits.device,
            dtype=torch.float
        )

        expected_class = torch.sum(probs * class_values, dim=-1)

        ordinal_loss = nn.functional.smooth_l1_loss(
            expected_class,
            labels.float()
        )

        loss = ce_loss + self.alpha * ordinal_loss

        return (loss, outputs) if return_outputs else loss

In [ ]:
def train_transformer_model_ordinal_loss(
    model_checkpoint,
    model_name,
    train_df,
    val_df,
    text_col=TEXT_COL,
    label_col=LABEL_COL,
    max_length=256,
    num_train_epochs=2,
    train_batch_size=16,
    eval_batch_size=32,
    learning_rate=2e-5,
    gradient_accumulation_steps=1,
    alpha=0.5
):
    """
    Train a transformer model for 10-class rating prediction using ordinal-aware loss.
    """
    num_classes = 10

    print(f"Training ordinal-aware model: {model_name}")
    print(f"Checkpoint: {model_checkpoint}")
    print(f"Text column: {text_col}")
    print(f"Label column: {label_col}")
    print(f"Alpha for ordinal penalty: {alpha}")

    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

    train_dataset = FlexibleDrugReviewTorchDataset(
        texts=train_df[text_col],
        labels=train_df[label_col],
        tokenizer=tokenizer,
        max_length=max_length
    )

    val_dataset = FlexibleDrugReviewTorchDataset(
        texts=val_df[text_col],
        labels=val_df[label_col],
        tokenizer=tokenizer,
        max_length=max_length
    )

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_checkpoint,
        num_labels=num_classes,
        use_safetensors=True
    )

    class_weights = compute_class_weights_from_labels(
        labels=train_df[label_col].values,
        num_classes=num_classes
    )

    output_dir = f"./{model_name.replace(' ', '_').replace('/', '_')}"

    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=learning_rate,
        per_device_train_batch_size=train_batch_size,
        per_device_eval_batch_size=eval_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        num_train_epochs=num_train_epochs,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        logging_steps=100,
        save_total_limit=1,
        report_to="none",
        seed=SEED,
        fp16=torch.cuda.is_available()
    )

    trainer_kwargs = {
        "model": model,
        "args": training_args,
        "train_dataset": train_dataset,
        "eval_dataset": val_dataset,
        "data_collator": data_collator,
        "compute_metrics": make_compute_metrics(num_classes),
        "class_weights": class_weights,
        "alpha": alpha,
        "num_classes": num_classes
    }

    try:
        trainer = OrdinalAwareTrainer(
            **trainer_kwargs,
            processing_class=tokenizer
        )
    except TypeError:
        trainer = OrdinalAwareTrainer(
            **trainer_kwargs,
            tokenizer=tokenizer
        )

    trainer.train()

    training_logs[model_name] = make_json_serializable(trainer.state.log_history)
    save_modeling_progress()

    predictions = trainer.predict(val_dataset)

    y_pred = np.argmax(predictions.predictions, axis=1)
    y_true = predictions.label_ids

    evaluate_model_flexible(
        model_name=model_name,
        y_true_labels=y_true,
        y_pred_labels=y_pred,
        num_classes=num_classes,
        target_names=[str(i) for i in range(1, 11)],
        task_name="10-class ordinal-aware rating"
    )

    plot_confusion_matrix_flexible(
        model_name=model_name,
        class_labels=list(range(1, 11))
    )

    training_logs[model_name] = make_json_serializable(trainer.state.log_history)
    save_modeling_progress()

    return trainer, tokenizer, model

In [ ]:
# RoBERTa with ordinal-aware loss

if not model_already_completed("RoBERTa Base + Ordinal Loss"):
    roberta_ordinal_trainer, roberta_ordinal_tokenizer, roberta_ordinal_model = train_transformer_model_ordinal_loss(
        model_checkpoint="roberta-base",
        model_name="RoBERTa Base + Ordinal Loss",
        train_df=train_df,
        val_df=val_df,
        text_col=TEXT_COL,
        label_col=LABEL_COL,
        max_length=256,
        num_train_epochs=2,
        train_batch_size=16,
        eval_batch_size=32,
        learning_rate=2e-5,
        gradient_accumulation_steps=1,
        alpha=0.5
    )
else:
    print("RoBERTa Base + Ordinal Loss already completed. Skipping.")

In [ ]:
# BERT with ordinal-aware loss

if not model_already_completed("BERT Base + Ordinal Loss"):
    bert_ordinal_trainer, bert_ordinal_tokenizer, bert_ordinal_model = train_transformer_model_ordinal_loss(
        model_checkpoint="bert-base-uncased",
        model_name="BERT Base + Ordinal Loss",
        train_df=train_df,
        val_df=val_df,
        text_col=TEXT_COL,
        label_col=LABEL_COL,
        max_length=256,
        num_train_epochs=2,
        train_batch_size=16,
        eval_batch_size=32,
        learning_rate=2e-5,
        gradient_accumulation_steps=1,
        alpha=0.5
    )
else:
    print("BERT Base + Ordinal Loss already completed. Skipping.")

In [ ]:
# BioClinicalBERT with ordinal-aware loss

if not model_already_completed("BioClinicalBERT + Ordinal Loss"):
    bioclinicalbert_ordinal_trainer, bioclinicalbert_ordinal_tokenizer, bioclinicalbert_ordinal_model = train_transformer_model_ordinal_loss(
        model_checkpoint="emilyalsentzer/Bio_ClinicalBERT",
        model_name="BioClinicalBERT + Ordinal Loss",
        train_df=train_df,
        val_df=val_df,
        text_col=TEXT_COL,
        label_col=LABEL_COL,
        max_length=256,
        num_train_epochs=2,
        train_batch_size=16,
        eval_batch_size=32,
        learning_rate=2e-5,
        gradient_accumulation_steps=1,
        alpha=0.5
    )
else:
    print("BioClinicalBERT + Ordinal Loss already completed. Skipping.")

In [ ]:
# DistilBERT with ordinal-aware loss

if not model_already_completed("DistilBERT + Ordinal Loss"):
    distilbert_ordinal_trainer, distilbert_ordinal_tokenizer, distilbert_ordinal_model = train_transformer_model_ordinal_loss(
        model_checkpoint="distilbert-base-uncased",
        model_name="DistilBERT + Ordinal Loss",
        train_df=train_df,
        val_df=val_df,
        text_col=TEXT_COL,
        label_col=LABEL_COL,
        max_length=256,
        num_train_epochs=2,
        train_batch_size=16,
        eval_batch_size=32,
        learning_rate=2e-5,
        gradient_accumulation_steps=1,
        alpha=0.5
    )
else:
    print("DistilBERT + Ordinal Loss already completed. Skipping.")

## 3-Class Sentiment Experiments Following the Published Paper

The published paper converted the original 10-star ratings into three sentiment classes.  
To make my work comparable to the paper, I performed the same conversion:

- ratings 1–4: negative
- ratings 5–6: neutral
- ratings 7–10: positive

This is not meant to replace the main 10-class rating prediction task.  
Instead, it provides an additional comparison point with the published work.

In [ ]:
# Create 3-class sentiment labels according to the mapping used in the paper

def rating_to_3class_label(rating):
    rating = int(rating)

    if rating <= 4:
        return 0
    elif rating <= 6:
        return 1
    else:
        return 2


def rating_to_3class_text(rating):
    label = rating_to_3class_label(rating)

    if label == 0:
        return "negative"
    elif label == 1:
        return "neutral"
    else:
        return "positive"


train_df["label_3class"] = train_df["rating"].apply(rating_to_3class_label)
val_df["label_3class"] = val_df["rating"].apply(rating_to_3class_label)

train_df["sentiment_3class"] = train_df["rating"].apply(rating_to_3class_text)
val_df["sentiment_3class"] = val_df["rating"].apply(rating_to_3class_text)

sentiment_target_names = ["negative", "neutral", "positive"]

print("Train 3-class distribution:")
display(
    train_df["sentiment_3class"]
    .value_counts()
    .rename_axis("sentiment")
    .reset_index(name="count")
)

print("Validation 3-class distribution:")
display(
    val_df["sentiment_3class"]
    .value_counts()
    .rename_axis("sentiment")
    .reset_index(name="count")
)

In [ ]:
# Weighted classification trainer for 3-class sentiment classification
# I use weighted cross-entropy because the sentiment classes are imbalanced.
# This follows the same general idea as the published paper, which handled class imbalance.

class WeightedClassificationTrainer(Trainer):
    """
    Hugging Face Trainer with weighted cross-entropy loss.
    """

    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.logits

        if self.class_weights is not None:
            weights = self.class_weights.to(logits.device)
            loss_fn = nn.CrossEntropyLoss(weight=weights)
        else:
            loss_fn = nn.CrossEntropyLoss()

        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [ ]:
def train_transformer_model_3class(
    model_checkpoint,
    model_name,
    train_df,
    val_df,
    text_col=TEXT_COL,
    label_col="label_3class",
    max_length=256,
    num_train_epochs=2,
    train_batch_size=16,
    eval_batch_size=32,
    learning_rate=2e-5,
    gradient_accumulation_steps=1
):
    """
    Train a transformer model for 3-class sentiment classification.
    This is used for comparison with the published paper.
    """
    num_classes = 3

    print(f"Training 3-class sentiment model: {model_name}")
    print(f"Checkpoint: {model_checkpoint}")
    print(f"Text column: {text_col}")
    print(f"Label column: {label_col}")

    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

    train_dataset = FlexibleDrugReviewTorchDataset(
        texts=train_df[text_col],
        labels=train_df[label_col],
        tokenizer=tokenizer,
        max_length=max_length
    )

    val_dataset = FlexibleDrugReviewTorchDataset(
        texts=val_df[text_col],
        labels=val_df[label_col],
        tokenizer=tokenizer,
        max_length=max_length
    )

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_checkpoint,
        num_labels=num_classes,
        use_safetensors=True
    )

    class_weights = compute_class_weights_from_labels(
        labels=train_df[label_col].values,
        num_classes=num_classes
    )

    output_dir = f"./{model_name.replace(' ', '_').replace('/', '_')}"

    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=learning_rate,
        per_device_train_batch_size=train_batch_size,
        per_device_eval_batch_size=eval_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        num_train_epochs=num_train_epochs,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        logging_steps=100,
        save_total_limit=1,
        report_to="none",
        seed=SEED,
        fp16=torch.cuda.is_available()
    )

    trainer_kwargs = {
        "model": model,
        "args": training_args,
        "train_dataset": train_dataset,
        "eval_dataset": val_dataset,
        "data_collator": data_collator,
        "compute_metrics": make_compute_metrics(num_classes),
        "class_weights": class_weights
    }

    try:
        trainer = WeightedClassificationTrainer(
            **trainer_kwargs,
            processing_class=tokenizer
        )
    except TypeError:
        trainer = WeightedClassificationTrainer(
            **trainer_kwargs,
            tokenizer=tokenizer
        )

    trainer.train()

    training_logs[model_name] = make_json_serializable(trainer.state.log_history)
    save_modeling_progress()

    predictions = trainer.predict(val_dataset)

    y_pred = np.argmax(predictions.predictions, axis=1)
    y_true = predictions.label_ids

    evaluate_model_flexible(
        model_name=model_name,
        y_true_labels=y_true,
        y_pred_labels=y_pred,
        num_classes=num_classes,
        target_names=sentiment_target_names,
        task_name="3-class sentiment"
    )

    plot_confusion_matrix_flexible(
        model_name=model_name,
        class_labels=sentiment_target_names
    )

    training_logs[model_name] = make_json_serializable(trainer.state.log_history)
    save_modeling_progress()

    return trainer, tokenizer, model

In [ ]:
# RoBERTa for 3-class sentiment classification

if not model_already_completed("RoBERTa Base 3-Class"):
    roberta_3class_trainer, roberta_3class_tokenizer, roberta_3class_model = train_transformer_model_3class(
        model_checkpoint="roberta-base",
        model_name="RoBERTa Base 3-Class",
        train_df=train_df,
        val_df=val_df,
        text_col=TEXT_COL,
        label_col="label_3class",
        max_length=256,
        num_train_epochs=2,
        train_batch_size=16,
        eval_batch_size=32,
        learning_rate=2e-5
    )
else:
    print("RoBERTa Base 3-Class already completed. Skipping.")

In [ ]:
# BERT for 3-class sentiment classification

if not model_already_completed("BERT Base 3-Class"):
    bert_3class_trainer, bert_3class_tokenizer, bert_3class_model = train_transformer_model_3class(
        model_checkpoint="bert-base-uncased",
        model_name="BERT Base 3-Class",
        train_df=train_df,
        val_df=val_df,
        text_col=TEXT_COL,
        label_col="label_3class",
        max_length=256,
        num_train_epochs=2,
        train_batch_size=16,
        eval_batch_size=32,
        learning_rate=2e-5
    )
else:
    print("BERT Base 3-Class already completed. Skipping.")

In [ ]:
# BioClinicalBERT for 3-class sentiment classification

if not model_already_completed("BioClinicalBERT 3-Class"):
    bioclinicalbert_3class_trainer, bioclinicalbert_3class_tokenizer, bioclinicalbert_3class_model = train_transformer_model_3class(
        model_checkpoint="emilyalsentzer/Bio_ClinicalBERT",
        model_name="BioClinicalBERT 3-Class",
        train_df=train_df,
        val_df=val_df,
        text_col=TEXT_COL,
        label_col="label_3class",
        max_length=256,
        num_train_epochs=2,
        train_batch_size=16,
        eval_batch_size=32,
        learning_rate=2e-5
    )
else:
    print("BioClinicalBERT 3-Class already completed. Skipping.")

In [ ]:
# DistilBERT for 3-class sentiment classification

if not model_already_completed("DistilBERT 3-Class"):
    distilbert_3class_trainer, distilbert_3class_tokenizer, distilbert_3class_model = train_transformer_model_3class(
        model_checkpoint="distilbert-base-uncased",
        model_name="DistilBERT 3-Class",
        train_df=train_df,
        val_df=val_df,
        text_col=TEXT_COL,
        label_col="label_3class",
        max_length=256,
        num_train_epochs=2,
        train_batch_size=16,
        eval_batch_size=32,
        learning_rate=2e-5
    )
else:
    print("DistilBERT 3-Class already completed. Skipping.")

In [ ]:
# Final save and reload after all additional experiments

save_modeling_progress()

loaded_results_df, loaded_confusion_matrices, loaded_training_histories, loaded_training_logs, loaded_interpretability_outputs = load_modeling_progress(
    save_dir=PROGRESS_DIR,
    restore_to_globals=True
)

results_df = show_results_table()
display(results_df)

## Final Test Set Prediction


In [ ]:
# Final Test Set Preprocessing - same preprocessing logic that was used earlier for the training/validation data

TEST_PATH = "drugsComTest_raw.csv"

test_raw = pd.read_csv(TEST_PATH)

print("Raw test shape:", test_raw.shape)
display(test_raw.head())


def preprocess_drug_reviews_test(df):
    """
    Apply the same preprocessing steps used for the training data.

    Steps:
    1. Copy the dataframe to avoid modifying the raw data directly.
    2. Fill missing condition values with 'Unknown'.
    3. Convert HTML entities into readable text.
    4. Normalize whitespace.
    5. Create review-only model input.
    6. Create metadata-enriched model input.
    7. Create label_10class if the rating column exists.
    8. Create log_usefulCount for consistency with the training preprocessing.
    """
    # Work on a copy to keep the original raw test set unchanged
    processed_df = df.copy()

    # Handle missing condition values - keep the row and mark missing condition as "Unknown"
    processed_df["condition"] = processed_df["condition"].fillna("Unknown")

    # Clean review text - convert HTML entities such as &quot; into normal characters
    processed_df["review_clean"] = processed_df["review"].astype(str).apply(html.unescape)

    # Normalize spaces, tabs, and line breaks
    def normalize_spaces(text):
        text = re.sub(r"\s+", " ", str(text))
        return text.strip()

    processed_df["review_clean"] = processed_df["review_clean"].apply(normalize_spaces)

    # Create model input text columns, review-only input for BERT Base
    processed_df["model_text_review_only"] = processed_df["review_clean"]

    # Metadata-enriched input for RoBERTa Base + Metadata
    processed_df["model_text_with_metadata"] = (
        "Drug: " + processed_df["drugName"].astype(str) +
        " | Condition: " + processed_df["condition"].astype(str) +
        " | Review: " + processed_df["review_clean"].astype(str) +
        " | usefulCount: " + processed_df["usefulCount"].astype(str)
    )

    # Create labels if rating exists in the test set
    if "rating" in processed_df.columns:
        processed_df["label_10class"] = processed_df["rating"].astype(int) - 1

    # Keep log usefulCount for consistency
    processed_df["log_usefulCount"] = np.log1p(processed_df["usefulCount"])

    return processed_df


# Apply preprocessing to the test set
test_df = preprocess_drug_reviews_test(test_raw)

test_df.to_csv("preprocessed_test.csv", index=False)

print("Preprocessed test shape:", test_df.shape)
print("Saved: preprocessed_test.csv")